### Human in the loop

In [25]:
from typing import Annotated
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from typing_extensions import TypedDict
from langchain_groq import ChatGroq

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.types import Command, interrupt
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [27]:
tavily = TavilySearch(max_results=2)

def human_feedback(query: str) -> str:
    """
    This function requests assistance from human, whenever assistance is required, this function must be called
    """
    human_res = interrupt({"query": query})
    return human_res["data"]

tools = [tavily, human_feedback]

llm = ChatGroq(model="openai/gpt-oss-120b")

llm_with_tools = llm.bind_tools(tools)
# llm_with_tools

In [28]:
# Node
def chatbot(state: State):
    system_message = SystemMessage(
        content=(
            "You are coordinating tools in a human-in-the-loop workflow. "
            "If the latest human feedback asks for web research or says to search the web, "
            "call the tavily tool with a focused search query. Do not call human_feedback again "
            "unless the user asks for more human assistance."
        )
    )
    message = llm_with_tools.invoke([system_message, *state["messages"]])
    return {"messages": [message]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

memory = MemorySaver()

graph = graph_builder.compile(checkpointer=memory)


In [29]:
# from IPython.display import display, Image

# display(Image(graph.get_graph().draw_mermaid_png()))

In [30]:
user_input = "I need some guidance and assistance for building AI agents. request assistance for me"
config = {"configurable": {"thread_id": "2"}}

events = graph.stream(
    {"messages": user_input},
    config,
    stream_mode="values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need some guidance and assistance for building AI agents. request assistance for me
================================== Ai Message ==================================
Tool Calls:
  human_feedback (fc_22fd1294-4a58-4ddc-b783-4c436679e3ee)
 Call ID: fc_22fd1294-4a58-4ddc-b783-4c436679e3ee
  Args:
    query: User requests guidance and assistance for building AI agents.
================================== Ai Message ==================================
Tool Calls:
  human_feedback (fc_22fd1294-4a58-4ddc-b783-4c436679e3ee)
 Call ID: fc_22fd1294-4a58-4ddc-b783-4c436679e3ee
  Args:
    query: User requests guidance and assistance for building AI agents.


In [31]:
human_response = (
    "We are the best experts and masters in building AI agents and we'd recommend you to checkout langgraph to build your agent"
    "Go ahead and search about AI agents on web"
)

human_command = Command(resume={"data": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_feedback (fc_22fd1294-4a58-4ddc-b783-4c436679e3ee)
 Call ID: fc_22fd1294-4a58-4ddc-b783-4c436679e3ee
  Args:
    query: User requests guidance and assistance for building AI agents.
================================= Tool Message =================================
Name: human_feedback

We are the best experts and masters in building AI agents and we'd recommend you to checkout langgraph to build your agentGo ahead and search about AI agents on web
================================== Ai Message ==================================
Tool Calls:
  tavily_search (fc_a3136efc-ce86-446f-be1c-01f34a1158c5)
 Call ID: fc_a3136efc-ce86-446f-be1c-01f34a1158c5
  Args:
    query: building AI agents guide langgraph tutorial
    search_depth: advanced
    time_range: year
================================= Tool Message =================================
Name: tavily_search

{"query": "building AI agents guide